## MCP

- A standard protocol that allows AI models to communicate with external tools and data sources in a structured, interoperable way.

- Break it down word by word:
- Model : The LLM (Claude, GPT, etc.)
- Context : Information the LLM can see and use
- Protocol : A set of rules for communication (like HTTP is a protocol for the web)

#### So literally: "Rules for how an AI model exchanges context with the outside world."

## How MCP Works

MCP acts like a **USB-C port for AI** — a universal, standardized way to plug any AI model into any external tool or data source without custom integrations.

---

## Core Components

### 1. MCP Host
- The application the user interacts with (e.g., Claude Desktop, Cursor, VS Code with AI extensions)
- Manages the overall session and coordinates between the user and the AI model

### 2. MCP Client
- Lives inside the Host
- Converts user requests into structured protocol messages
- Maintains a **1:1 relationship** with a single MCP Server
- Multiple clients can exist within one Host

### 3. MCP Server
- External service or process that exposes capabilities to the LLM
- Connects to databases, APIs, file systems, web services, etc.
- Responds to requests from the MCP Client

---

## Core Primitives (What a Server Can Expose)

| Primitive | Description | Example |
|-----------|-------------|---------|
| **Tools** | Actions the AI can trigger | Run a query, send an email, search the web |
| **Resources** | Structured data the AI can read | Files, database records, documentation |
| **Prompts** | Predefined instruction templates | "Summarize this repo", "Review this PR" |

---

## Communication Protocol

MCP uses **JSON-RPC 2.0** for all client-server communication.

### Transport Mechanisms
- **Stdio (Standard I/O)** — for local processes on the same machine; fast and simple
- **Streamable HTTP** — for remote servers; uses HTTP POST + Server-Sent Events (SSE); supports auth

### Session Lifecycle
```
1. initialize    →  Client connects and sends capabilities
2. negotiate     →  Server responds with its own capabilities
3. ready         →  Both sides confirm, session is active
4. request/respond  →  Normal tool/resource/prompt exchanges
5. shutdown      →  Session cleanly terminated
```

---

## End-to-End Flow (Example)

```
User: "What files changed in the last commit?"

[Host] → receives user message
  ↓
[MCP Client] → formats it as a JSON-RPC request
  ↓
[MCP Server (Git Tool)] → runs `git log -1 --name-only`
  ↓
[MCP Client] → receives result
  ↓
[LLM] → processes result and generates a response
  ↓
User sees: "The following files changed: main.py, utils.py..."
```

---

## Why MCP Matters

- **No more custom glue code** — one protocol works across all AI models and tools
- **Interoperability** — any MCP-compatible host can connect to any MCP-compatible server
- **Open standard** — donated to the Linux Foundation (Agentic AI Foundation) in December 2025
- **Scalable** — easily add new tools/resources without changing the AI model

---

> **In short:** MCP standardizes how AI models get context from the outside world — making AI integrations modular, reusable, and model-agnostic.

## Simple MCP Server — Addition Tool

A minimal MCP server that exposes one tool: `add(a, b)`.  
Run it with `python addition_server.py` — it communicates over **stdio** (standard input/output).

In [1]:
# addition_server.py  —  Simple MCP server with an addition tool

server_code = '''
from mcp.server.fastmcp import FastMCP

# Create the MCP server and give it a name
mcp = FastMCP("Addition Server")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b


if __name__ == "__main__":
    mcp.run()   # runs over stdio by default
'''

# Write the server to a file so it can be run directly
with open("addition_server.py", "w") as f:
    f.write(server_code.strip())

print("addition_server.py created successfully!")

addition_server.py created successfully!


### Test the tool directly (in-process)

No need to spin up the server just to verify the logic — call the function directly.

In [2]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Addition Server")

@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

# Test the tool directly
print(add(3, 5))      # → 8.0
print(add(10.5, 4.5)) # → 15.0
print(add(-2, 7))     # → 5.0

8
15.0
5


### How to connect a client to this server

```python
# client.py — connect to the addition server via stdio
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def main():
    server_params = StdioServerParameters(
        command="python",
        args=["addition_server.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Call the add tool
            result = await session.call_tool("add", {"a": 10, "b": 25})
            print("Result:", result.content[0].text)  # → 35.0

asyncio.run(main())
```

> Run `python client.py` in the terminal after the server file is created.

---

## MCP Calculator — with User Input

A full calculator MCP server exposing **add, subtract, multiply, divide** as tools.  
The client takes values from the user via `input()` and calls the matching tool on the server.

### Step 1 — Calculator Server (`calculator_server.py`)

In [3]:
server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Calculator Server")


@mcp.tool()
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@mcp.tool()
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@mcp.tool()
def divide(a: float, b: float) -> float:
    """Divide a by b. Raises error if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero.")
    return a / b


if __name__ == "__main__":
    mcp.run()
'''

with open("calculator_server.py", "w") as f:
    f.write(server_code.strip())

print("calculator_server.py created!")

calculator_server.py created!


### Step 2 — Calculator Client (`calculator_client.py`)

Takes user input, launches the server as a subprocess over stdio, and calls the right tool.

In [ ]:
client_code = '''
import asyncio
import os
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "calculator_server.py")

OPERATIONS = {
    "1": ("add",      "+"),
    "2": ("subtract", "-"),
    "3": ("multiply", "*"),
    "4": ("divide",   "/"),
}

async def run_calculator():
    server_params = StdioServerParameters(
        command="python",
        args=[SERVER_PATH]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            print("\\n=============================")
            print("   MCP Calculator")
            print("=============================")

            while True:
                print("\\nSelect operation:")
                for key, (name, symbol) in OPERATIONS.items():
                    print(f"  {key}. {name.capitalize()} ({symbol})")
                print("  5. Exit")

                choice = input("\\nEnter choice (1-5): ").strip()

                if choice == "5":
                    print("Goodbye!")
                    break

                if choice not in OPERATIONS:
                    print("Invalid choice. Try again.")
                    continue

                try:
                    a = float(input("Enter first number : "))
                    b = float(input("Enter second number: "))
                except ValueError:
                    print("Invalid number. Please enter digits only.")
                    continue

                tool_name, symbol = OPERATIONS[choice]

                try:
                    result = await session.call_tool(tool_name, {"a": a, "b": b})
                    answer = result.content[0].text
                    print(f"\\nResult: {a} {symbol} {b} = {answer}")
                except Exception as e:
                    print(f"Error: {e}")

asyncio.run(run_calculator())
'''

with open("calculator_client.py", "w") as f:
    f.write(client_code.strip())

print("calculator_client.py created!")

### How to Run

1. **Execute both cells above** to generate the two `.py` files.
2. Open a terminal and run:
   ```bash
   python calculator_client.py
   ```
3. You'll see an interactive menu like:

```
=============================
   MCP Calculator
=============================

Select operation:
  1. +
  2. -
  3. *
  4. /
  5. Exit

Enter choice (1-5): 3
Enter first number : 6
Enter second number: 7

Result: 6.0 * 7.0 = 42.0
```

> The client **automatically starts the server** as a subprocess — you only need to run `calculator_client.py`.

### What's happening under the hood

```
calculator_client.py
  │
  ├─ spawns ──► calculator_server.py  (subprocess over stdio)
  │                  │
  │             FastMCP receives JSON-RPC call
  │             runs add / subtract / multiply / divide
  │             returns result over stdout
  │
  └─ prints result to user
```

---

## IMDB Top 100 Movie Recommendation MCP Server

An MCP server with the **IMDB Top 100 movies** embedded as data.  
Exposes 5 tools — search by genre, director, title, or get smart recommendations.

| Tool | What it does |
|------|-------------|
| `get_top_movies(limit)` | Show top N movies from the list |
| `search_by_genre(genre)` | Filter movies by genre (Drama, Action, Sci-Fi…) |
| `search_by_director(director)` | All movies by a director |
| `get_movie_details(title)` | Full details with partial title match |
| `recommend_movies(genre, min_rating, limit)` | Smart filtered recommendations |

> Source: [IMDb Top 250](https://www.imdb.com/chart/top/) — top 100 entries with ratings, genres, and directors.

### How to Run

```bash
python "c:\ALL Project Files\MCP\MCP\movie_client.py"
```

**Sample session:**
```
╔══════════════════════════════════════╗
║     IMDB Top 100 Movie Recommender   ║
╠══════════════════════════════════════╣
║  1. Show Top N Movies                ║
║  2. Search by Genre                  ║
║  3. Search by Director               ║
║  4. Get Movie Details                ║
║  5. Get Recommendations              ║
║  6. Exit                             ║
╚══════════════════════════════════════╝

Enter choice (1-6): 5
Enter genre to filter (leave blank for all): Sci-Fi
Minimum IMDB rating (e.g. 8.5): 8.5
How many recommendations? (e.g. 5): 3

Top 3 Recommendations in Sci-Fi (min rating 8.5)
========================================
# 13  Inception (2010)  ⭐8.8  [Action, Adventure, Sci-Fi]  — Christopher Nolan
# 25  Interstellar (2014)  ⭐8.6  [Adventure, Drama, Sci-Fi]  — Christopher Nolan
# 45  Alien (1979)  ⭐8.5  [Horror, Sci-Fi]  — Ridley Scott
```

---

# MCP Complete Knowledge Guide

> This section goes deeper — covering every MCP primitive, security model, agentic patterns, and the full 2025–2026 evolution of the protocol.

---

## MCP Primitives — Deep Dive

MCP defines **four core primitives** that a server can expose. Most tutorials only cover Tools — but Resources, Prompts, and Sampling are equally important.

---

### 1. Tools (AI-controlled actions)

The LLM decides when to call a tool. Tools perform actions or fetch dynamic data.

```python
@mcp.tool()
def get_stock_price(ticker: str) -> str:
    """Fetch the live price of a stock ticker."""
    ...
```

- **Input**: typed parameters the LLM passes
- **Output**: text or structured data returned to the LLM
- **Control**: LLM decides when to invoke

---

### 2. Resources (application-controlled data)

Resources are **readable data** that the host app (not the LLM) exposes into context.  
Think of them like files or database rows the AI can read.

```python
@mcp.resource("file://{path}")
def read_file(path: str) -> str:
    """Expose a local file as a readable MCP resource."""
    with open(path) as f:
        return f.read()
```

- **URI-addressed**: identified by a URI scheme (e.g. `file://`, `db://`, `git://`)
- **Control**: the host/application decides what resources to expose
- **Use cases**: files, database rows, config, documentation

---

### 3. Prompts (user-controlled templates)

Prompts are **predefined instruction templates** the user can invoke directly (like slash commands).

```python
@mcp.prompt()
def code_review(language: str, code: str) -> str:
    """Review code for bugs and style issues."""
    return f"Review this {language} code for bugs:\n\n{code}"
```

- **Control**: the user explicitly selects which prompt to run
- **Use cases**: "Summarize this PR", "Review this SQL query", "Explain this error"

---

### 4. Sampling (server asks LLM for completions)

Sampling lets the **server request LLM completions** — flipping the usual direction.  
This enables agentic loops where the server itself reasons and acts.

```python
# Server-side: ask the host LLM to generate text
result = await ctx.sample(
    messages=[{"role": "user", "content": "Summarize this data: ..."}],
    max_tokens=200
)
```

- **Direction**: Server → Client → LLM (reversed from normal)
- **Use cases**: agentic pipelines, server-side reasoning, multi-step workflows
- **Security**: the host must approve all sampling requests (human-in-the-loop)

---

### Primitive Control Matrix

| Primitive | Controlled by | Direction | Example |
|-----------|--------------|-----------|---------|
| **Tools** | LLM | Client → Server | Run a query, search the web |
| **Resources** | Application | Server → Context | Read a file, load DB row |
| **Prompts** | User | User → Client | Slash command templates |
| **Sampling** | Server | Server → LLM | Server-initiated reasoning |

---

## MCP Security & Authorization

### OAuth 2.1 (Mandatory since June 2025 spec)

As of the **2025-06-18 spec revision**, MCP servers are classified as **OAuth resource servers**.  
Clients must use **Resource Indicators (RFC 8707)** to prevent token theft by rogue servers.

```
User
 │
 ▼
MCP Client  ──── OAuth 2.1 ────►  Authorization Server
                                        │
                                        │ access_token
                                        ▼
MCP Client  ──── Bearer Token ────►  MCP Server (Resource Server)
```

**Why it matters:**  
Before mandatory OAuth, a malicious MCP server could request tokens scoped for another service. Resource Indicators lock the token to the specific server that requested it.

---

### Elicitation (2025-11-25 spec)

Elicitation lets a **server define what structured input it needs** from the client, including secure out-of-band authorization flows.

**URL Mode Elicitation (SEP-1036):**
1. Server signals it needs third-party authorization (e.g. GitHub OAuth)
2. Client opens the URL in the user's browser
3. User completes the OAuth flow in-browser
4. The third-party service delivers tokens **directly to the server** — the client never sees them

```
MCP Server ──elicit(url)──► MCP Client ──opens browser──► GitHub OAuth
                                                                │
                                                     token delivered directly
                                                                ▼
                                                          MCP Server ✓
```

This avoids token passthrough — the client never handles third-party credentials.

---

### Structured Tool Outputs (2025-11-25 spec)

Tools can now return **machine-readable objects** instead of plain text, enabling safe tool chaining.

```python
# Before: tools returned strings
return "Temperature is 22°C"

# After: tools return typed objects
return {"temperature": 22, "unit": "celsius", "feels_like": 20}
```

The downstream tool or LLM can reliably parse and act on the structured result without string parsing.

---

## Multi-Agent & Agentic Patterns with MCP

MCP is increasingly used not just as a tool-calling protocol but as the backbone for **multi-agent systems** where agents delegate to other agents.

---

### Agent-to-Agent via MCP

An MCP server can itself be an **agent** that runs its own reasoning loop.  
Using sampling, a server can ask the host LLM for guidance mid-execution.

```
User
 │
 ▼
Orchestrator Agent (MCP Client)
 │
 ├──► Weather MCP Server  (fetches data)
 ├──► DB MCP Server       (queries records)
 └──► Sub-Agent MCP Server
         │
         └──► uses Sampling to ask LLM to reason about the data
```

---

### Tasks Primitive (Experimental)

The **Tasks primitive** ships as experimental in the 2025–2026 roadmap.  
It models long-running, asynchronous operations with a full lifecycle:

| State | Meaning |
|-------|---------|
| `pending` | Task created, not yet started |
| `running` | Actively executing |
| `completed` | Finished successfully |
| `failed` | Errored — retry semantics apply |
| `expired` | Result TTL elapsed, result discarded |

```python
# Server exposes a long-running task
@mcp.task()
async def run_etl_pipeline(source: str, destination: str) -> TaskResult:
    """Run a multi-hour ETL pipeline and report completion."""
    ...
```

The client polls or subscribes to task status rather than blocking on the result.

---

### Agentic Design Patterns

| Pattern | Description | When to use |
|---------|-------------|-------------|
| **Tool Chaining** | Output of tool A feeds into tool B | Sequential data pipelines |
| **Parallel Tools** | Multiple tools called simultaneously | Independent data sources |
| **Server-side Sampling** | Server asks LLM mid-execution | Complex reasoning within a tool |
| **Hierarchical Agents** | Orchestrator delegates to sub-agents via MCP | Large, decomposable tasks |
| **Human-in-the-loop** | Elicitation pauses for user approval | Sensitive / destructive actions |

---

## Recent Advancements in MCP (2025–2026)

### Timeline of Key Milestones

```
Nov 2024  ──  Anthropic open-sources MCP
Mar 2025  ──  OpenAI officially adopts MCP (ChatGPT desktop)
Mar 2025  ──  Streamable HTTP transport replaces SSE
Jun 2025  ──  OAuth 2.1 becomes mandatory; structured tool outputs land
Jul 2025  ──  SEP (Spec Enhancement Proposal) process launched
Nov 2025  ──  1-year anniversary spec; elicitation, server identity, async ops
Dec 2025  ──  Anthropic donates MCP to the Linux Foundation (Agentic AI Foundation)
2026      ──  Focus: enterprise readiness, Tasks primitive, transport scalability
```

---

### 1. Streamable HTTP Transport (March 2025)

Replaced the original SSE (Server-Sent Events) mechanism with a proper bi-directional HTTP transport:

- Single HTTP connection for both sending and receiving
- Chunked transfer encoding for progressive message delivery
- Scales to remote, stateless server deployments
- Enables MCP servers behind standard load balancers

---

### 2. OpenAI Adoption (March 2025)

OpenAI integrated MCP across its products including the **ChatGPT desktop app**.  
This made MCP truly model-agnostic — not just a Claude/Anthropic feature — cementing it as an industry standard.

---

### 3. OAuth 2.1 + Structured Outputs (June 2025)

- Servers are now **OAuth resource servers** — tokens are locked to the server via RFC 8707
- Tools return **typed JSON objects**, not just strings → enables reliable tool chaining
- Prevents token theft by rogue servers in multi-server setups

---

### 4. SEP Process & Community Governance (July 2025)

Anthropic introduced the **Specification Enhancement Proposal (SEP)** process — like Python's PEPs but for MCP.  
Working groups now drive the spec openly rather than one company controlling it.

---

### 5. November 2025 Spec — Major Feature Release

The anniversary release packed several features:

| Feature | What it adds |
|---------|-------------|
| **Elicitation** | Servers can request structured user input / OAuth flows |
| **Server Identity** | Servers can prove who they are (prevents impersonation) |
| **Async Operations** | Long-running server ops without blocking |
| **Official Registry** | Community-driven, searchable MCP server registry |
| **Statelessness Support** | Servers can be fully stateless for horizontal scaling |

---

### 6. Donated to Linux Foundation (December 2025)

MCP is now governed by the **Agentic AI Foundation (AAIF)**, a directed fund under the Linux Foundation, co-founded by Anthropic, OpenAI, and Block.

This means:
- No single company controls the spec
- Enterprise confidence in long-term stability
- Governed like HTTP or OAuth — as open infrastructure

---

### 7. 2026 Roadmap Priorities

| Priority | Problem being solved |
|----------|---------------------|
| **Transport Scalability** | Stateful sessions conflict with horizontal scaling and load balancers |
| **Tasks Primitive (GA)** | Retry semantics, expiry policies for long-running async tasks |
| **Enterprise Readiness** | Audit trails, SSO-integrated auth, gateway behavior, config portability |
| **Governance Maturation** | More working groups, broader community contribution |

---

### By the Numbers (as of early 2026)

| Metric | Value |
|--------|-------|
| Monthly SDK downloads | **97 million** (Python + TypeScript) |
| Active MCP servers | **10,000+** |
| Registered servers (mcpservers.org) | **23,000+** |
| Companies with official MCP servers | Microsoft, Google, Atlassian, Stripe, GitHub, AWS, and more |

---

## MCP Ecosystem & Discovery

### Where to Find MCP Servers

| Registry | URL | Notes |
|----------|-----|-------|
| **Official Registry** | modelcontextprotocol.io | Curated, maintained by AAIF |
| **mcpservers.org** | mcpservers.org | Largest — 23,000+ servers |
| **Awesome MCP Servers** | github.com/punkpeye/awesome-mcp-servers | Community curated list |
| **mcp.so** | mcp.so | Searchable catalog |

### MCP Servers by Category

| Category | Popular Servers |
|----------|----------------|
| **Databases** | PostgreSQL, MySQL, SQLite, MongoDB, Supabase, Neon |
| **Code & Dev** | GitHub, GitLab, Jira, Linear, Sentry |
| **Files** | Local filesystem, Google Drive, Dropbox, Notion |
| **Search** | Tavily, Brave Search, Exa |
| **Communication** | Slack, Gmail, Outlook |
| **Cloud** | AWS, GCP, Azure, Vercel |
| **AI Models** | Anthropic, OpenAI, Groq, Replicate |
| **Finance** | Stripe, QuickBooks, Plaid |

---

## FastMCP Quick Reference

```python
from mcp.server.fastmcp import FastMCP, Context

mcp = FastMCP("My Server")

# Tool — LLM-invoked action
@mcp.tool()
def my_tool(param: str) -> str:
    """Tool description shown to the LLM."""
    return f"Result for {param}"

# Resource — application-managed readable data
@mcp.resource("data://{name}")
def my_resource(name: str) -> str:
    """Expose data as a readable resource."""
    return f"Data: {name}"

# Prompt — user-invoked instruction template
@mcp.prompt()
def my_prompt(topic: str) -> str:
    """Instruction template for the LLM."""
    return f"Explain {topic} clearly and concisely."

# Tool with context (for logging, sampling, etc.)
@mcp.tool()
async def advanced_tool(query: str, ctx: Context) -> str:
    await ctx.info(f"Processing: {query}")   # log to client
    result = await ctx.sample(               # ask LLM mid-execution
        messages=[{"role": "user", "content": query}]
    )
    return result.content

if __name__ == "__main__":
    mcp.run()   # stdio (local) — default
    # mcp.run(transport="streamable-http", host="0.0.0.0", port=8000)  # remote
```

---

## MCP vs Other Integration Approaches

| Approach | Flexibility | Reusability | Standardized | AI-native |
|----------|------------|-------------|--------------|-----------|
| **Custom API wrapper** | High | Low (per-model) | No | No |
| **LangChain tools** | Medium | Medium | Partial | Yes |
| **OpenAI function calling** | Medium | Low (GPT only) | No | Yes |
| **MCP** | High | High (any model) | Yes (open spec) | Yes |

> **Key insight:** MCP is the only approach where the same server works with Claude, GPT, Gemini, and any future model without changes.

---

## Summary — What Makes MCP Different

```
Without MCP:
  Claude ──custom──► Tool A
  Claude ──custom──► Tool B
  GPT    ──custom──► Tool A   ← you write this integration again
  Gemini ──custom──► Tool A   ← and again

With MCP:
  Any LLM ──MCP──► Any Tool   ← write once, works everywhere
```

MCP does for AI integrations what **HTTP did for the web** and **USB-C did for devices** —  
a single standard that makes everything interoperable.